In [ ]:
from pymilvus import MilvusClient
from pymilvus import connections, MilvusClient, DataType
from pymilvus import AnnSearchRequest

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless
TOKEN = "YOUR_MILVUS_TOKEN"

client = MilvusClient(
    uri=CLUSTER_ENDPOINT,
    token=TOKEN 
)

client.list_collections()

## RAG案例

In [ ]:
import pandas as pd
from tqdm import tqdm

repo_name = "cloudstack"
local_file_path = 'dataset/mergebert/vector/' + repo_name + '_Conflicts.txt'
# local_file_path = 'dataset/50repo/vector/' + repo_name + '_Conflicts.txt'

# 使用 pandas 读取本地文件
pair = pd.read_csv(local_file_path, sep='\t', header=None).drop_duplicates()
conflicts = pair[0].values
res_labels = pair[1].values
global_id = pair[2].values

data_pairs = list(zip(conflicts, res_labels, global_id))

# 打印前几行，查看数据是否正确读取
print(conflicts.size) 
print(conflicts[:3])
print(res_labels[:3]) 
print(global_id[:3])

In [ ]:
client.drop_collection(collection_name=repo_name)

schema = MilvusClient.create_schema(
    auto_id=True,
    enable_dynamic_field=True,
)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="global_id", datatype=DataType.INT32)
schema.add_field(field_name="chunk_content", datatype=DataType.VARCHAR, max_length=2048)
schema.add_field(field_name="chunk_resolution", datatype=DataType.VARCHAR, max_length=1024)
schema.add_field(field_name="chunk_sparse_embedding", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="chunk_embedding", datatype=DataType.FLOAT_VECTOR, dim=768)

index_params = MilvusClient.prepare_index_params()
index_params.add_index(
    field_name="chunk_embedding",
    metric_type="COSINE",
    index_type="AUTOINDEX",
    index_name="vector_index"
)
index_params.add_index(
    field_name="chunk_sparse_embedding",
    index_name="sparse_inverted_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP",
    params={"drop_ratio_build": 0.2},
)


client.create_collection(
    collection_name=repo_name, 
    schema=schema, 
    index_params=index_params
)

In [ ]:
from pymilvus.model.sparse.bm25.tokenizers import build_default_analyzer
from pymilvus.model.sparse import BM25EmbeddingFunction
 
analyzer = build_default_analyzer(language="en") 
bm25_ef = BM25EmbeddingFunction(analyzer)
bm25_ef.fit(conflicts)

In [ ]:
"""注释掉:已改成codebert模型嵌入"""
# from sentence_transformers import SentenceTransformer
# local_model_path = "./bge/bge-small-en-v1.5"

# # # 在线加载模型
# # from sentence_transformers import SentenceTransformer
# # model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm

# 加载本地 CodeBERT 模型
local_model_path = "./codebert/codebert-base"

# 加载模型和分词器
def download_or_load_model(local_model_path):
    try:
        # 尝试从本地加载模型
        tokenizer = AutoTokenizer.from_pretrained(local_model_path)
        model = AutoModel.from_pretrained(local_model_path).to("cuda" if torch.cuda.is_available() else "cpu")
        print("使用本地模型")
    except Exception as e:
        # 捕获并打印错误，帮助调试
        print(f"从本地加载模型失败，错误：{e}")
        
        # 本地不存在则从网络下载
        tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
        model = AutoModel.from_pretrained("microsoft/codebert-base").to("cuda" if torch.cuda.is_available() else "cpu")
        
        # 保存到本地
        tokenizer.save_pretrained(local_model_path)
        model.save_pretrained(local_model_path)
        print("从网络下载并保存本地模型")

    return tokenizer, model

tokenizer, model = download_or_load_model(local_model_path)

# 截取或者填充到 768 维
# def ensure_vector_dimension(vector, target_dim=768):
#     if len(vector) > target_dim:
#         return vector[:target_dim]
#     elif len(vector) < target_dim:
#         return np.pad(vector, (0, target_dim - len(vector)), 'constant')
#     return vector

for conflict, resolution, g_id in tqdm(data_pairs, desc="Processing conflicts and resolutions"):
    # 使用 tokenizer 对文本进行编码
    inputs = tokenizer(conflict, padding=True, truncation=True, return_tensors="pt").to(model.device)
    
    # 使用 CodeBERT 获取嵌入（通过取最后一层的均值作为句子嵌入）
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # 获取句子级别的 embedding
    
    encode = embeddings[0]
    result = bm25_ef.encode_queries([conflict]).reshape(1, -1)
     # 处理 resolution 字段
    if not isinstance(resolution, str) or not resolution.strip():
        res_data = "_newline_"
    else:
        res_data = str(resolution).strip()

    res = client.insert(
        collection_name=repo_name,
        data=[
            {
                "global_id": g_id,
                "chunk_content": conflict.encode('utf-8')[:2048].decode('utf-8', errors='ignore'),
                "chunk_resolution": res_data.encode('utf-8')[:1024].decode('utf-8', errors='ignore'),
                "chunk_sparse_embedding": {index: value for index, value in zip(result.indices, result.data)},
                "chunk_embedding": encode
            }
        ]
    )


In [ ]:
import matplotlib.pyplot as plt

with open(local_file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

part1_lengths = []
part2_lengths = []

for line in tqdm(lines):
    # 使用 \t 分割每行
    parts = line.strip().split('\t', 1)
    
    if len(parts) == 2:  
        part1_length = len(parts[0].strip())  # 计算第一部分的字符长度
        part2_length = len(parts[1].strip())  # 计算第二部分的字符长度
        
        part1_lengths.append(part1_length)
        part2_lengths.append(part2_length)

plt.figure(figsize=(14, 6))

# 第一部分
plt.subplot(1, 2, 1)
plt.hist(part1_lengths, bins=100, color='#3498db', edgecolor='black')  # 使用蓝色
plt.title("Character Length Distribution of Conflict")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")
plt.xlim(0, 4000)  # 根据实际需要调整x轴范围
plt.grid(True)

# 第二部分
plt.subplot(1, 2, 2)
plt.hist(part2_lengths, bins=100, color='#e74c3c', edgecolor='black')  # 使用红色
plt.title("Character Length Distribution of Resolution")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")
plt.xlim(0, 2000)  # 根据实际需要调整x轴范围
plt.grid(True)

plt.tight_layout()  # 自动调整子图参数，使之填充整个图像区域
plt.show()

In [ ]:
"""绘制token数量分布的柱状图"""
# import matplotlib.pyplot as plt

# # 读取 output.txt 文件并获取每行文本
# with open(local_file_path, "r", encoding="utf-8") as file:
#     lines = file.readlines()

# # 统计每行的token数量
# token_counts = []
# for line in tqdm(lines):
#     # 对每行文本进行tokenization
#     inputs = tokenizer(line.strip(), padding=True, truncation=True, return_tensors="pt")
#     token_count = len(inputs["input_ids"][0])  # 获取token数量
#     token_counts.append(token_count)

# plt.figure(figsize=(10, 6))
# plt.hist(token_counts, bins=30, color='skyblue', edgecolor='black')  # 20个bins
# plt.title("Token Length Distribution")
# plt.xlabel("Number of Tokens")
# plt.ylabel("Frequency")
# plt.xlim(0, 600)  # 根据实际需要调整x轴范围
# plt.grid(True)
# plt.show()

In [ ]:
from pymilvus import AnnSearchRequest, WeightedRanker

query = "import static org.junit.Assert.assertTrue; _newline_  _newline_  _newline_ base_content _newline_  _newline_ incoming_change _newline_ import java.text.SimpleDateFormat;"
result = bm25_ef.encode_queries([query]).reshape(1, -1)

sparse_search_params = {"metric_type": "IP"}
sparse_req = AnnSearchRequest([{index: float(value) for index, value in zip(result.indices, result.data)}],
                              "chunk_sparse_embedding", sparse_search_params, limit=2)

# 对查询使用 CodeBERT 进行编码（得到密集向量）
inputs = tokenizer(query, padding=True, truncation=True, return_tensors="pt").to(model.device)
with torch.no_grad():  # 禁用梯度计算
    outputs = model(**inputs)
    dense_embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# 确保密集向量的维度为 768
dense_embedding = dense_embedding[0]

dense_search_params = {"metric_type": "COSINE"}
dense_req = AnnSearchRequest([dense_embedding], "chunk_embedding", dense_search_params, limit=2)

reqs = [sparse_req, dense_req]
rerank = WeightedRanker(0.2, 0.8)
  
result = client.hybrid_search(repo_name, reqs, ranker=rerank, limit=3, output_fields=["chunk_content",  "chunk_resolution", "global_id"])

In [ ]:
related_conflict = "\n".join([x["entity"]["chunk_content"] for x in result[0]])
related_resolution = "\n".join([str(x["entity"]["chunk_resolution"]) for x in result[0]])
global_id = "\n".join([str(x["entity"]["global_id"]) for x in result[0]])
prompt = f"""查询：
{query}

检索结果：
{related_conflict}

对应resolution:
{related_resolution}

全局id:
{global_id}

"""
print(prompt)